# Lektion 12 - Reduzierung des Chatverlaufs mit dem Agent Scratchpad

Dieses Notebook zeigt, wie man den Kontext in langen Gesprächen mit dem Microsoft Agent Framework verwaltet. Mit zunehmender Länge der Gespräche steigt die Anzahl der Tokens — was schließlich das Kontextfenster des Modells überschreitet. Wir lösen dies mit einem **Kontext-Zusammenfassungs-Pattern** und einem **Agent Scratchpad** für ein persistentes Gedächtnis.

## Was Sie lernen werden:
1. **Warum Kontextmanagement wichtig ist**: Verständnis von Token-Limits und Kontextfenstern
2. **Kontextbewusste Agenten**: Aufbau von Agenten, die ihren eigenen Gesprächskontext verwalten
3. **Kontext-Zusammenfassungs-Pattern**: Nutzung von Tools zur Verdichtung des Gesprächsverlaufs
4. **Agent Scratchpad**: Persistentes Gedächtnis, das die Kontextreduktion übersteht

## Voraussetzungen:
- Azure OpenAI Einrichtung mit konfigurierten Umgebungsvariablen
- Verständnis grundlegender Agentenkonzepte aus vorherigen Lektionen


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Warum Kontextmanagement wichtig ist

Jedes LLM hat ein begrenztes **Kontextfenster** — die maximale Anzahl von Tokens, die es in einer einzelnen Anfrage verarbeiten kann. Im Verlauf eines mehrstufigen Gesprächs:

- **Wächst die Token-Anzahl linear** mit jeder Benutzer- und Assistenten-Nachricht.
- **Bestimmen Prompt-Tokens die Kosten**, weil der gesamte Verlauf bei jedem Durchlauf erneut gesendet wird.
- Irgendwann überschreitet das Gespräch das **Kontextfenster** und das Modell kürzt oder gibt einen Fehler aus.

### Strategien für das Management des Kontexts

| Strategie | Funktionsweise | Kompromiss |
|---|---|---|
| **Abschneiden** | Älteste Nachrichten verwerfen | Frühere Kontextinformationen gehen verloren |
| **Zusammenfassung** | Ältere Nachrichten in einer Zusammenfassung verdichten | Einige Details gehen verloren, aber Kernpunkte bleiben erhalten |
| **Notizblock / Externer Speicher** | Wichtige Fakten außerhalb des Gesprächs speichern | Erfordert Tool-Aufrufe, übersteht aber jede Reduzierung |

In diesem Notebook kombinieren wir **Zusammenfassung** mit einem **Notizblock-Tool**, damit der Agent Kontinuität bewahren kann, selbst wenn der Gesprächsverlauf verdichtet wird.


## Erstellen eines kontextbewussten Agenten


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulation eines langen Gesprächs

Lassen Sie uns ein mehrtüriges Gespräch durchgehen, um zu sehen, wie sich der Kontext aufbaut. Der Agent sollte wichtige Details (Vorlieben, Budget, Reisedaten) über die Gesprächsrunden hinweg behalten und Kontinuität zeigen.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Beachten Sie, wie der Agent den Kontext aus früheren Gesprächsrunden behält – er weiß über Japan, Sushi, Tempel, Fotografie, das Budget von 3000 $, Alleinreisen und den Zeitraum im April Bescheid. In einem kurzen Gespräch funktioniert das gut, aber je länger das Gespräch wird, desto aufwändiger wird es, die gesamte Historie erneut zu senden.

Lassen Sie uns das Gespräch mit weiteren Runden fortsetzen, um die Anhäufung von Kontext zu sehen:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontext-Zusammenfassungsmuster

Wenn das Gespräch wächst, können wir ein **Zusammenfassungstool** verwenden, um den angesammelten Kontext in ein kompaktes Format zu verdichten. Der Agent ruft dieses Tool auf, um wichtige Präferenzen zu speichern, sodass auch bei Wegfall älterer Nachrichten die wesentlichen Informationen erhalten bleiben.

Dieses Muster ist der Baustein für eine ausgefeiltere Verlaufsreduktion:
1. Der Agent identifiziert Schlüsselfakten aus dem Gespräch
2. Er ruft das Zusammenfassungstool auf, um sie zu speichern
3. Ältere Nachrichten können sicher entfernt werden, da die Zusammenfassung das Wesentliche erfasst

Im Folgenden definieren wir ein `summarize_preferences`-Tool, das der Agent aufrufen kann, um eine kompakte Zusammenfassung dessen, was er gelernt hat, zu erstellen.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man den Kontext in langfristigen Agenten-Gesprächen mithilfe des Microsoft Agent Framework verwaltet:

### Wichtige Konzepte
- **Kontextfenster sind begrenzt** — jedes Token im Gesprächsverlauf kostet Geld und zählt zum Limit.
- **Zusammenfassungstools** ermöglichen es dem Agenten, angesammelten Kontext in kompakte Zusammenfassungen zu kondensieren, wodurch die Token-Nutzung reduziert und wesentliche Informationen erhalten bleiben.
- **Agenten-Scratchpads** bieten einen persistierenden externen Speicher, der jede Gesprächsverkürzung überlebt.

### Was Sie gebaut haben
- Einen **kontextbewussten Agenten**, der Kontinuität über mehrstufige Gespräche hinweg aufrechterhält
- Ein **Zusammenfassungstool** (`summarize_preferences`), das wichtige Benutzerdetails kompakt speichert
- Ein **mehrstufiges Gespräch**, das Kontextbeibehaltung und Umgang mit Änderungen demonstriert

### Anwendungen in der Praxis
- **Kundendienst-Bots**: Erinnern an Präferenzen über lange Support-Sitzungen hinweg
- **Persönliche Assistenten**: Verfolgen laufender Projekte ohne erneute Kontext-Erklärung
- **Bildungstutoren**: Halten den Fortschritt der Schüler über viele Interaktionen aufrecht

### Nächste Schritte
- Implementieren Sie ein vollständiges Scratchpad-Tool mit dateibasierter Persistenz
- Fügen Sie automatische Verlaufsverkürzung nach Zusammenfassung hinzu
- Kombinieren Sie mit Vektordatenbanken für semantische Suchfunktion im Speicher
- Entwickeln Sie Agenten, die Gespräche auch nach Tagen mit vollem Kontext wieder aufnehmen können


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
